# Sprint 6 — Model Comparison (the blast finish)

Every model we built, baseline vs `class_weight='balanced'`, on the same held-out test (566,126 featured flows). The metrics are from the verified training runs (a kernel-SVM predict on 566k takes 10+ min, so the RBF rows are recorded, not re-run here). The deployable models live in `models/`.

Model names: **Inertia** = Decision Tree · **Graphite** = LinearSVC · **Graphite 2.0 / 2.1 / 2.2** = RBF at 50k / 250k / 500k.

In [ ]:
import pandas as pd

rows = [
    # name, model, class_weight, acc, recall, precision, FN, FP, train
    ('Inertia',      'Decision Tree',      'none',     0.9990, 0.9984, 0.9967,   178,   369, '38s'),
    ('Inertia',      'Decision Tree',      'balanced', 0.9991, 0.9988, 0.9968,   134,   357, '35s'),
    ('Graphite',     'LinearSVC (2.26M)',  'none',     0.9412, 0.8875, 0.8268, 12550, 20733, '~4m'),
    ('Graphite',     'LinearSVC (2.26M)',  'balanced', 0.9225, 0.9603, 0.7307,  4431, 39469, '~5m'),
    ('Graphite 2.0', 'RBF 50k',            'none',     0.9629, 0.8809, 0.9273, 13281,  7704, '~2m'),
    ('Graphite 2.0', 'RBF 50k',            'balanced', 0.9466, 0.9807, 0.7956,  2150, 28094, '~2m'),
    ('Graphite 2.1', 'RBF 250k',           'balanced', 0.9665, 0.9779, 0.8686,  2469, 16500, '~14m'),
    ('Graphite 2.2', 'RBF 500k',           'none',     0.9704, 0.9190, 0.9300,  9030,  7712, '~35m'),
    ('Graphite 2.2', 'RBF 500k',           'balanced', 0.9677, 0.9755, 0.8749,  2732, 15554, '~35m'),
]
cols = ['name', 'model', 'class_weight', 'accuracy', 'recall', 'precision', 'FN(missed)', 'FP(false_alarm)', 'train_time']
df = pd.DataFrame(rows, columns=cols)
df

## Verdict

**Inertia (the Decision Tree) wins, decisively and cheaply.** It is the only model that gets high recall *and* high precision at the same time (0.9988 recall, 0.9968 precision, ~134 missed attacks and ~357 false alarms), trained in **38 seconds** on all 2.26M rows, with no tuning.

Everything else has to trade:
- **Graphite (LinearSVC):** a straight boundary can't fit the data. `class_weight` lifts recall (0.89 → 0.96) but floods false alarms (20.7k → 39.5k).
- **Graphite 2.x (RBF):** the curved boundary helps, and more data helps recall (0.88 → 0.92 at 500k). `class_weight` pushes recall to ~0.98, but every SVM buys recall with a false-alarm flood, and at brutal compute (500k RBF ≈ 35 min vs the tree's 38 s).

Two headline lessons:
1. **`class_weight` beat data:** RBF-50k+cw (0.98 recall) beat RBF-500k baseline (0.92). The untreated imbalance hurt recall more than data-starvation did.
2. **The tree's inductive bias fits network-flow data** — threshold splits mirror how attacks are actually defined; the SVM's smooth margin fights that shape.

**Deployment:** Inertia is the model you'd ship (fast to train, near-instant inference, best balance). The Graphite line is the comparison story; for the SVM slots the *baseline* configs are the better-balanced choice (`class_weight`'s false-alarm flood isn't worth it here).

## Reading a change: the delta ratio (the evaluation lens)

When a change (class_weight, more data) moves the numbers, weigh **attacks caught (ΔFN)** against **false alarms added (ΔFP)**. The **delta ratio = false alarms added per attack caught** (ΔFP / ΔFN), judged against your security cost ratio (how many false alarms one missed intrusion is worth):
- ratio <= 0 -> **strict win** (caught more *and* fewer alarms)
- ratio < 1 -> favorable (more attacks caught than alarms added)
- ratio >> 1 -> costly (many alarms per attack; only worth it if a miss costs that much more)

The tell: **more data is near-free recall** (better boundary, no alarm cost), while **class_weight is traded recall** (paid in false alarms), and that trade gets cheaper on a better boundary.

In [ ]:
# each row: a change, and its measured (attacks caught, false alarms added) on the test set
changes = [
    ('Tree  +class_weight',          44,   -15),
    ('LinearSVC  +class_weight',   8119, 18736),
    ('RBF-50k  +class_weight',    11131, 20390),
    ('RBF-500k  +class_weight',    6298,  7842),
    ('RBF  50k->500k (more data)',  4251,     8),
]
dd = pd.DataFrame(changes, columns=['change', 'attacks_caught', 'alarms_added'])
dd['alarms_per_attack'] = (dd['alarms_added'] / dd['attacks_caught']).round(3)
dd['verdict'] = dd['alarms_per_attack'].map(
    lambda r: 'strict win' if r <= 0 else ('favorable' if r < 1 else 'costly trade'))
dd

In [ ]:
import joblib
from pathlib import Path

mdir = Path('../models')
print('deployable models saved in models/:')
for f in sorted(mdir.glob('*.joblib')):
    print(' ', f.name)